<a href="https://colab.research.google.com/github/roasfora/weather_api_facultet_ds4/blob/main/Instructor_Weather_API_Workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Instructor edition — Weather API → pandas → CSV
**Module 2 · Files, APIs, and Data Sources**  
**Suggested duration: 50–60 minutes**

**Learning objectives:** students can describe a GET request, parse JSON, build a pandas DataFrame, create a calculated column, export a CSV, and explain workflow vs pipeline.

**Classroom pacing:** 5 min concept → 10 min `requests` → 10 min inspect JSON → 10 min pandas → 5 min CSV → 10 min independent tasks → 5 min pipeline discussion.

The forecast is live; values differ each day. Uses [Open-Meteo](https://open-meteo.com/en/docs), no API key for non-commercial classroom use.

## Talk track: workflow vs pipeline

- **Workflow:** the steps or process (collect → transform → save). It can be manual.
- **Pipeline:** the same steps implemented to run together, repeatedly, with little manual intervention.
- A notebook with separately executed cells is a **workflow**. A function that performs the full process with one call is a small **pipeline**.

**Analogy:** workflow = recipe; pipeline = kitchen machine that runs the recipe.

In [ ]:
%pip install requests pandas

## Collect — HTTP GET and JSON
Show how `params` creates a query string; explain 200 status, `.raise_for_status()`, and Python dictionary.

In [ ]:
# Open-Meteo gives us weather forecasts as JSON.
# We request 7 days of weather for Lisbon (no API key needed).
import requests
import pandas as pd

url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 41.15,
    "longitude": -8.61,
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
    "timezone": "Europe/Lisbon",
    "forecast_days": 7,
}


In [ ]:
response = requests.get(url, params=params, timeout=20)
response.raise_for_status()  # Show an error if the request failed.
data = response.json()        # Convert the JSON response to Python data.
print("HTTP status:", response.status_code)
print("Top-level JSON keys:", list(data.keys()))


In [ ]:
print(data)

In [ ]:
url_2 = "https://api.open-meteo.com/v1/forecast_"
response = requests.get(url_2, params=params, timeout=20)
response.raise_for_status()  # Show an error if the request failed.
data = response.json()        # Convert the JSON response to Python data.
print("HTTP status:", response.status_code)
print("Top-level JSON keys:", list(data.keys()))

In [ ]:
print("Daily keys:", list(data["daily"].keys()))
print("First date:", data["daily"]["time"][0])
print("First max temperature:", data["daily"]["temperature_2m_max"][0])

## Transform — DataFrame and new column
**Demo prompt:** Why is it easier to analyze a table than several separate Python lists?

In [ ]:
# The 'daily' part contains one list per column.
df = pd.DataFrame(data["daily"])
print(df)




In [ ]:
# Give columns short, readable names.
df = df.rename(columns={
    "time": "date",
    "temperature_2m_max": "max_c",
    "temperature_2m_min": "min_c",
    "precipitation_sum": "rain_mm",
})

# A simple new column (transformation).
df["temperature_range_c"] = df["max_c"] - df["min_c"]
df.head()

In [ ]:
print("Rows and columns:", df.shape)
print("Column names:", df.columns.tolist())
df.describe()

## Save — reusable CSV file

In [ ]:
filename = "lisbon_weather.csv"
df.to_csv(filename, index=False)
print("Saved:", filename)


In [ ]:
check = pd.read_csv(filename)
assert len(check) == 7
assert "temperature_range_c" in check.columns
print("CSV verified:", check.shape)
check.head()

## Exercise answers (students work independently first)

In [ ]:
display(df)
rainy_days = df[df["rain_mm"] > 0]
display(rainy_days)
rainy_days.to_csv("rainy_days.csv", index=False)
print("Saved rainy_days.csv — rows:", len(rainy_days))

## Optional extension — from workflow to pipeline

This is intentionally **after** the basic exercise. `run_pipeline()` wraps the same three actions in one reusable function. Emphasize that beginner students do **not** need to understand advanced architecture or scheduling yet.

In [ ]:
def run_pipeline(city, latitude, longitude):
    # Collect
    parameters = {
        "latitude": latitude,
        "longitude": longitude,
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
        "timezone": "auto",
        "forecast_days": 7,
    }
    response = requests.get(url, params=parameters, timeout=20)
    response.raise_for_status()
    weather = response.json()["daily"]

    # Transform
    table = pd.DataFrame(weather).rename(columns={
        "time": "date",
        "temperature_2m_max": "max_c",
        "temperature_2m_min": "min_c",
        "precipitation_sum": "rain_mm",
    })
    table["temperature_range_c"] = table["max_c"] - table["min_c"]

    # Save
    file_path = f"{city.lower()}_weather.csv"
    table.to_csv(file_path, index=False)
    return file_path, table

path, result = run_pipeline("Lisbon", 38.72, -9.14)
print("Pipeline created:", path)
result.head()

## Common pitfalls and answers

- **No internet / timeout:** API requests need a working connection; retry later.
- **`ModuleNotFoundError`:** run the install cell.
- **`KeyError`:** inspect `data.keys()` and `data["daily"].keys()`; confirm the parameters.
- **CSV seems missing:** it is saved in the notebook's current working directory; in Colab, use the Files panel to download it.
- **Rainy days is empty:** that is valid if the forecast has no rain.
- **Forecast numbers change:** data is live, so assess structure and logic, not exact values.

**Exit ticket answers:** API supplies data; `.json()` converts a JSON response into Python objects; DataFrame rename/new-column operations transform; `.to_csv()` saves; manually run cells = workflow, one-call function = simple pipeline.

**Licensing:** Credit Open-Meteo (weather data CC BY 4.0); the free hosted API is for non-commercial use, subject to provider terms.